In [28]:
from sqlalchemy import text
from config.database import engine

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
        print("✅ Connection OK")
except Exception as e:
    print("❌ Connection FAILED")
    print(e)

✅ Connection OK


In [29]:
from sqlalchemy import text
from config.database import engine
from services.load_data import init_db
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public';
    """))

    for t in result:
        print("✔", t[0])

init_db()

✔ segments
✔ clients
✔ produits
✔ agences
✔ temps_transaction
🚀 Starting database initialization...

✅ segments created
✔ segments created
✅ clients created
✔ clients created
✅ produits created
✔ produits created
✅ agences created
✔ agences created
✅ temps_transaction created
✔ temps_transaction created
❌ ERROR while creating tables:
(psycopg2.errors.SyntaxError) ERREUR:  erreur de syntaxe sur ou près de « , »
LINE 4:         transaction_id varchar(50) UNIQUE NOT NULL,,
                                                           ^

[SQL: 
    CREATE TABLE IF NOT EXISTS transactions (
        id_transaction SERIAL PRIMARY KEY,
        transaction_id varchar(50) UNIQUE NOT NULL,,
        compte_id INT NOT NULL,
        produit_id INT NOT NULL,
        agence_id INT NOT NULL,
        temps_id INT NOT NULL,

        montant FLOAT NOT NULL ,
        taux_change_eur FLOAT NOT NULL ,

        devise VARCHAR(10) NOT NULL,
        type_operation VARCHAR(50) NOT NULL,
        statut VARCHAR(50) N

les indexes

In [30]:
from models.index import create_indexes
create_indexes()

ProgrammingError: (psycopg2.errors.UndefinedTable) ERREUR:  la relation « transactions » n'existe pas

[SQL: 
            CREATE INDEX IF NOT EXISTS idx_transactions_agence
            ON transactions(agence_id);
        ]
(Background on this error at: https://sqlalche.me/e/20/f405)

Chargement des Données via SQLAlchemy

In [ ]:
import pandas as pd

df = pd.read_csv("../data/financecore_clean.csv")

df.head()

,transaction_id,client_id,date_transaction,montant,devise,taux_change_eur,montant_eur,categorie,produit,agence,...,statut,score_credit_client,segment_client,solde_avant,is_anomaly,annees,mois,trimestre,jour_semaine,taux_rejet
0,TXN000559,CLI0060,2022-04-19 02:31:00,2050.42,EUR,1.00,2050.42,Depot especes,Compte Epargne,Marseille-Vieux-Port,...,Complete,643.0,Premium,16415.10,False,2022,4,2,1,5.405405
1,TXN001154,CLI0057,2024-06-20 20:51:00,-123.66,GBP,0.86,-143.79,Retrait DAB,Credit Consommation,Bordeaux-Meriadeck,...,Rejete,435.0,Risque,42890.81,False,2024,6,2,3,4.428904
2,TXN000764,CLI0015,2024-08-28 05:03:00,-396.17,EUR,1.00,-396.17,Prelevement,PEA,Lyon-Part-Dieu,...,Complete,648.0,Standard,48489.38,False,2024,8,3,2,4.193548
3,TXN001598,CLI0045,2024-01-07 08:16:00,225.20,EUR,1.00,225.20,Paiement CB,Credit Consommation,Bordeaux-Meriadeck,...,Complete,704.0,Standard,43962.51,False,2024,1,1,6,4.428904
4,TXN001873,CLI0034,2024-08-11 19:52:00,935.32,EUR,1.00,935.32,Interets,Credit Immobilier,Bordeaux-Meriadeck,...,Complete,457.0,Risque,17312.83,False,2024,8,3,6,4.428904


Séparer financecore_clean.csv selon les tables normalisées avant insertion

In [ ]:
print(df.columns)

Index(['transaction_id', 'client_id', 'date_transaction', 'montant', 'devise',
       'taux_change_eur', 'montant_eur', 'categorie', 'produit', 'agence',
       'type_operation', 'statut', 'score_credit_client', 'segment_client',
       'solde_avant', 'is_anomaly', 'annees', 'mois', 'trimestre',
       'jour_semaine', 'taux_rejet'],
      dtype='object')


In [ ]:


# =========================
# 1. SEGMENTS
# =========================
segments_df = df[['segment_client']].drop_duplicates().reset_index(drop=True)
segments_df['segment_id'] = segments_df.index + 1

# =========================
# 2. CLIENTS
# =========================
clients_df = df[['client_id', 'score_credit_client', 'segment_client']].drop_duplicates()

clients_df = clients_df.merge(
    segments_df,
    on='segment_client',
    how='left'
)

clients_df = clients_df[['client_id', 'score_credit_client', 'segment_id']]

# =========================
# 3. COMPTES (optional dimension)
# =========================
comptes_df = df[['client_id', 'solde_avant']].drop_duplicates().reset_index(drop=True)
comptes_df['compte_id'] = comptes_df.index + 1

# IMPORTANT: avoid merging full df (memory fix)
clients_to_compte = comptes_df[['client_id', 'compte_id']]

df = df.merge(
    clients_to_compte,
    on='client_id',
    how='left'
)

# =========================
# 4. AGENCES
# =========================
agences_df = df[['agence', 'taux_rejet']].drop_duplicates().reset_index(drop=True)
agences_df['agence_id'] = agences_df.index + 1

# =========================
# 5. PRODUITS
# =========================
produits_df = df[['produit', 'categorie']].drop_duplicates().reset_index(drop=True)
produits_df['produit_id'] = produits_df.index + 1

# =========================
# 6. TEMPS
# =========================
df = df.rename(columns={"annees": "annee"})

temps_df = df[['date_transaction', 'annee', 'mois', 'trimestre', 'jour_semaine']].drop_duplicates()
temps_df = temps_df.reset_index(drop=True)
temps_df['temps_id'] = temps_df.index + 1

# =========================
# 7. TRANSACTIONS FACT TABLE
# =========================
transactions_df = df.copy()

# JOIN AGENCE
transactions_df = transactions_df.merge(
    agences_df[['agence', 'agence_id']],
    on='agence',
    how='left'
)

# JOIN PRODUIT
transactions_df = transactions_df.merge(
    produits_df[['produit', 'produit_id']],
    on='produit',
    how='left'
)

# JOIN TEMPS
transactions_df = transactions_df.merge(
    temps_df[['date_transaction', 'temps_id']],
    on='date_transaction',
    how='left'
)

# =========================
# FINAL TABLE (FIXED)
# =========================
transactions_df = transactions_df[[
    'transaction_id',
    'client_id',        # ✔ correct
    'compte_id',        # ✔ comes from merge
    'agence_id',
    'produit_id',
    'temps_id',
    'montant',
    'devise',
    'taux_change_eur',
    'montant_eur',
    'type_operation',
    'statut',
    'is_anomaly'
]]

MemoryError: Unable to allocate 530. MiB for an array with shape (10, 6944545) and data type object

Insérer les données

In [ ]:
def load_data():
    # =========================
    # DIMENSIONS (SAFE ORDER)
    # =========================

    segments_df.to_sql("segments", engine, if_exists="replace", index=False)

    clients_df.to_sql("clients", engine, if_exists="replace", index=False)

    comptes_df.to_sql("comptes", engine, if_exists="replace", index=False)

    agences_df.to_sql("agences", engine, if_exists="replace", index=False)

    produits_df.to_sql("produits", engine, if_exists="replace", index=False)

    temps_df.to_sql("temps_transaction", engine, if_exists="replace", index=False)

    # =========================
    # FACT TABLE
    # =========================
    transactions_df.to_sql(
        "transactions",
        engine,
        if_exists="replace",   # ✔ مهم باش ما يبقاش duplicate
        index=False
    )

    print("🚀 Data loaded successfully (CLEAN RELOAD)")

In [ ]:
from models.view import create_views
create_views()